[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jeevash23/logogram/blob/main/notebooks/attribution-patching.ipynb)

# Estimate every head at once, then check the estimates by patching

In [ ]:
%pip install logogram

Activation patching runs the model once per site. **Attribution patching** estimates all of them
from gradients: the change patching a site would cause is approximated as (source activation −
receiver activation) · the gradient of the metric (Nanda, 2023; Syed, Rager and Conmy, 2023,
"Attribution Patching Outperforms Automated Circuit Discovery"). A whole sweep then costs a few
passes, but the estimate is first order: it misses saturation in attention, normalization and
the softmax, and can miss or even invert an effect. **Integrated gradients** (Sundararajan, Taly
and Yan, 2017) average the gradient along the way from one prompt's embeddings to the other's,
as EAP-IG does (Hanna, Pezzelle and Belinkov, 2024), which corrects much of that.

So an estimate is a list of candidates: this notebook verifies the strongest by patching them
for real, then shows where a single gradient goes wrong.

## Open the example and load the model

In [ ]:
import logogram as lg

project = lg.open_example("ioi-gpt2")
model = lg.load_model("openai-community/gpt2")
patching = lg.load_spec(project.root / "experiments/ioi-head-patching/spec.json")

## Estimate every head

The example's spec patches every head. The same spec with an attribution patching experiment
estimates them instead, here with integrated gradients over 8 steps.

In [ ]:
estimate = lg.run(
    lg.load_spec(
        {
            **patching.model_dump(),
            "name": "Every head, estimated with integrated gradients",
            "experiment": {
                "kind": "attribution_patching",
                "direction": "clean_to_corrupt",
                "method": "integrated_gradients",
                "steps": 8,
            },
        }
    ),
    project,
    model=model,
)
estimate

## Verify the strongest by patching

`verify_run` patches the ten heads with the largest estimated effects, for real, in a run of its
own that records which run it verifies.

In [ ]:
checked = lg.verify_run(estimate, top=10)
estimated = {site["label"]: site["effect"]["mean"] for site in estimate.summary["sites"]}
print(f"{'head':<8}{'estimated':>10}{'patched':>9}  95% interval")
for site in checked.strongest(10):
    e = site["effect"]
    print(f"{site['label']:<8}{estimated[site['label']]:>+10.3f}{e['mean']:>+9.3f}  [{e['lo']:+.3f}, {e['hi']:+.3f}]")
comparison = lg.compare_runs(estimate, checked)
print(f"\nRank correlation between estimate and patching: {comparison['spearman']:.2f}")
print("Signs that patching confidently reverses:", comparison["n_sign_changes"])

## Where one gradient goes wrong

Head outputs track patching closely. The residual stream is harder: patching it replaces a whole
token's representation, far from where a single gradient was taken. Estimate the residual stream
before each layer at the prompts' named positions with one gradient, then patch every one of
those sites for real with `check_robustness`.

In [ ]:
stream = {
    **patching.model_dump(),
    "scope": {"kind": "layer_position", "site": "resid_pre", "positions": "labels"},
}
one_gradient = lg.run(
    lg.load_spec(
        {
            **stream,
            "name": "Residual stream, estimated with one gradient",
            "experiment": {
                "kind": "attribution_patching",
                "direction": "clean_to_corrupt",
                "method": "gradient",
                "steps": None,
            },
        }
    ),
    project,
    model=model,
)
patched = lg.check_robustness(
    one_gradient, experiment={"kind": "activation_patching", "direction": "clean_to_corrupt"}
)
print(f"Rank correlation with patching: {lg.compare_runs(one_gradient, patched)['spearman']:.2f}")

Integrated gradients follow the metric along the way. With 16 steps, their estimates of the same
sites agree with patching much more closely.

In [ ]:
integrated = lg.run(
    lg.load_spec(
        {
            **stream,
            "name": "Residual stream, estimated with integrated gradients",
            "experiment": {
                "kind": "attribution_patching",
                "direction": "clean_to_corrupt",
                "method": "integrated_gradients",
                "steps": 16,
            },
        }
    ),
    project,
    model=model,
)
print(f"Rank correlation with patching: {lg.compare_runs(integrated, patched)['spearman']:.2f}\n")
gradient = {site["label"]: site["effect"]["mean"] for site in one_gradient.summary["sites"]}
steps = {site["label"]: site["effect"]["mean"] for site in integrated.summary["sites"]}
print(f"{'site':<22}{'patched':>9}{'1 gradient':>12}{'16 steps':>10}")
for site in patched.strongest(6):
    label = site["label"]
    print(f"{label:<22}{site['effect']['mean']:>+9.3f}{gradient[label]:>+12.3f}{steps[label]:>+10.3f}")

An estimate decides what to look at; patching decides what is true. In the app, **Verify top 10
by patching** on an attribution patching run's results does what `verify_run` does here.